# Customer Segmentation with RFM and K-Means

**Objective:** Create RFM features, select clusters with the elbow method, profile segments and recommend marketing actions.

> Dataset note: included CSVs are synthetic practice data created for this portfolio. Replace with approved public datasets for official internship submission.


## Setup
Run the following cell. Install missing packages with `pip install -r ../../requirements.txt` from the repository root.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pathlib import Path
df=pd.read_csv('customer_transactions_synthetic.csv',parse_dates=['InvoiceDate'])
print(df.shape); display(df.head()); print(df.isna().sum())
snapshot=pd.Timestamp('2026-01-01')
rfm=df.groupby('Customer_ID').agg(Recency=('InvoiceDate',lambda x:(snapshot-x.max()).days),
 Frequency=('InvoiceDate','count'),Monetary=('Amount','sum'))
display(rfm.describe())
features=['Recency','Frequency','Monetary']
X=StandardScaler().fit_transform(rfm[features])
inertia=[]
for k in range(2,9):
    inertia.append(KMeans(n_clusters=k,random_state=42,n_init=10).fit(X).inertia_)
plt.plot(range(2,9),inertia,marker='o'); plt.title('Elbow method'); plt.xlabel('K'); plt.ylabel('Inertia'); plt.show()
model=KMeans(n_clusters=4,random_state=42,n_init=10)
rfm['Cluster']=model.fit_predict(X)
sns.scatterplot(data=rfm,x='Recency',y='Monetary',hue='Cluster',palette='tab10'); plt.show()
sns.scatterplot(data=rfm,x='Frequency',y='Monetary',hue='Cluster',palette='tab10'); plt.show()
display(rfm.groupby('Cluster')[features].mean().round(2))
rfm.Cluster.value_counts().sort_index().plot(kind='bar',title='Customers per cluster'); plt.show()
print('Marketing: low-recency/high-value customers → loyalty rewards; high-recency customers → win-back offers; frequent buyers → subscriptions; low-value occasional buyers → bundles.')
